# Glassbox LLMs — End-to-End Demo

This notebook demonstrates the core interpretability experiments in the **glassbox-llms** library. Each section loads a model, runs an experiment, and visualizes the results.

We use GPT-2 (124M parameters) throughout for consistency.

## 0. Setup

Load GPT-2 once and share it across all experiments.

In [ ]:
# pip install glassbox-llms
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import AutoModel, AutoTokenizer, GPT2LMHeadModel

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no pad token by default

model = AutoModel.from_pretrained(model_name)
model.eval()

# LM head model (needed for logit lens + causal patching)
lm_model = GPT2LMHeadModel.from_pretrained(model_name)
lm_model.eval()
lm_model.tokenizer = tokenizer  # PatchingExperiment reads model.tokenizer

print(f"Model: {model_name}")
print(f"Layers: {model.config.n_layer}, Hidden dim: {model.config.n_embd}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

---
## 1. Linear Probing — Where Does GPT-2 Encode Sentiment?

Train a linear classifier on frozen activations at each layer. If it can decode sentiment, that layer has a linear representation of it.

In [ ]:
from glassboxllms.primitives.probes import LinearProbe
from glassboxllms.primitives.probes.activation_store import ActivationStore
from glassboxllms.visualization.plots import plot_probe_accuracy

store = ActivationStore(model)
layers = [f"h.{i}" for i in range(model.config.n_layer)]

# Sentiment-labeled data — mix of clear and subtle examples
train_texts = [
    "I absolutely love this movie, it was fantastic!",
    "What a wonderful experience, highly recommended!",
    "The food was delicious and the service was excellent.",
    "Amazing performance, the actors were brilliant!",
    "The sunset was breathtaking, absolutely gorgeous.",
    "The presentation went well overall.",
    "She smiled when she heard the good news.",
    "The team celebrated after winning the championship.",
    "I hate this movie, it was terrible and boring.",
    "What a horrible experience, never going back!",
    "The food was disgusting and the service was awful.",
    "Terrible performance, the actors were embarrassing.",
    "The weather was miserable and depressing all week.",
    "He frowned at the disappointing results.",
    "The project failed to meet any of its goals.",
    "Traffic was awful and I missed my flight.",
]
train_labels = [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]

# Harder test set — subtler sentiment, less overlap with training
test_texts = [
    "The concert exceeded all my expectations.",
    "I found the book quite engaging and thought-provoking.",
    "The garden looked lovely in the morning light.",
    "We had a pleasant conversation over coffee.",
    "I was let down by the quality of the product.",
    "The restaurant was noisy and the food was cold.",
    "I struggled to stay awake during the lecture.",
    "The whole trip was a series of unfortunate events.",
]
test_labels = [1, 1, 1, 1, 0, 0, 0, 0]
print(f"Data: {len(train_texts)} train, {len(test_texts)} test | Layers: {layers}")

In [ ]:
# Train a probe at each layer and collect results
results = {}
best_layer, best_acc = None, 0

for layer in layers:
    train_acts = store.extract(train_texts, tokenizer, layers=[layer], pooling="mean")
    test_acts = store.extract(test_texts, tokenizer, layers=[layer], pooling="mean")

    probe = LinearProbe(layer=layer, direction="sentiment", model_type="logistic", normalize=True)
    probe.fit(train_acts[layer], np.array(train_labels))
    result = probe.evaluate(test_acts[layer], np.array(test_labels))
    results[layer] = result

    if result.accuracy > best_acc:
        best_layer, best_acc = layer, result.accuracy
    print(f"  {layer}: accuracy={result.accuracy:.3f}, f1={result.f1:.3f}")

print(f"\nBest layer: {best_layer} (accuracy={best_acc:.3f})")

In [ ]:
# Visualize probe accuracy across layers
fig = plot_probe_accuracy(results, metric="accuracy")
fig.suptitle("Linear Probe: Sentiment Classification by Layer (GPT-2)", fontsize=14, y=1.02)
plt.show()

Early layers encode syntax/position, later layers encode semantics like sentiment — probe accuracy increases across layers, confirming sentiment becomes **linearly decodable** in GPT-2's later layers.

---
## 2. CoT Faithfulness — Does the Model Actually Use Its Reasoning?

Two tests: (1) **Truncation** — cut reasoning short and see if the answer changes. If yes, the model was actually using it. (2) **Error injection** — insert a wrong answer into the reasoning. If the model follows it, it reads its own chain-of-thought.

In [ ]:
from glassboxllms.experiments.cot_faithfulness import CoTFaithfulnessEvaluator
import os

# Load API key from .env file (TOGETHER_API_KEY=your_key)
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass  # pip install python-dotenv if needed

from openai import OpenAI
client = OpenAI(
    api_key=os.environ.get("TOGETHER_API_KEY"),
    base_url="https://api.together.xyz/v1",
)

def generate_fn(prompt: str) -> str:
    response = client.chat.completions.create(
        model="meta-llama/Llama-3.3-70B-Instruct-Turbo",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300,
        temperature=0.7,
    )
    return response.choices[0].message.content

print("Model ready: Llama-3.3-70B via Together AI")

In [ ]:
# Run faithfulness evaluation (small n_samples for demo speed)
evaluator = CoTFaithfulnessEvaluator()
cot_result = evaluator.evaluate(
    generate_fn=generate_fn,
    model_name="Llama-3.3-70B",
    dataset="arc",
    n_samples=5,
    verbose=True,
)
print(cot_result.summary())

In [ ]:
# Visualize: truncation vs error injection scores + baseline comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: scores breakdown
scores = [cot_result.truncation_faithfulness, cot_result.error_following, cot_result.avg_faithfulness]
labels = ["Truncation", "Error Following", "Average"]
colors = ["#4C72B0", "#DD8452", "#55A868"]
axes[0].bar(labels, scores, color=colors, edgecolor="black", linewidth=0.5)
axes[0].set_ylim(0, 105)
axes[0].set_ylabel("Score (%)")
axes[0].set_title(f"CoT Faithfulness: {cot_result.model_name}")
for i, v in enumerate(scores):
    axes[0].text(i, v + 2, f"{v:.0f}%", ha="center", fontweight="bold")

# Right: compare to baseline
comparison = evaluator.compare_to_baseline(cot_result, baseline="Llama-3.1-70B")
if comparison:
    baseline_avg = cot_result.avg_faithfulness - comparison["difference"]
    models = [cot_result.model_name, "Llama-3.1-70B\n(baseline)"]
    avgs = [cot_result.avg_faithfulness, baseline_avg]
    axes[1].bar(models, avgs, color=["#4C72B0", "#999999"], edgecolor="black", linewidth=0.5)
    axes[1].set_ylim(0, 105)
    axes[1].set_ylabel("Avg Faithfulness (%)")
    axes[1].set_title(f"vs Baseline ({comparison['interpretation']})")
    for i, v in enumerate(avgs):
        axes[1].text(i, v + 2, f"{v:.0f}%", ha="center", fontweight="bold")
else:
    axes[1].text(0.5, 0.5, "No baseline available", ha="center", va="center", transform=axes[1].transAxes)

fig.tight_layout()
plt.show()

High **truncation** score = the model's answer depends on its reasoning (faithful). High **error following** = the model reads and trusts its own CoT. Larger models aren't necessarily more faithful — architecture and training matter more.

---
## 3. SAE — Decomposing GPT-2 into Sparse Features

A Sparse Autoencoder learns to represent a layer's activations as a sparse combination of interpretable feature directions. We train one on layer 8 activations and inspect which features fire and how sparsely.

In [ ]:
from glassboxllms.primitives.probes.activation_store import ActivationStore
from glassboxllms.features.sae import SparseAutoencoder
from glassboxllms.features.trainer import SAETrainer
from torch.utils.data import DataLoader, TensorDataset

# 1. Extract activations from layer h.8 (full sequence, no pooling)
sae_store = ActivationStore(model)
sae_texts = [
    "The cat sat on the mat and purred softly.",
    "Scientists discovered a new species deep in the ocean.",
    "She opened the door and stepped into the sunlight.",
    "The stock market crashed after the announcement.",
    "He picked up the guitar and played a familiar tune.",
    "Rain poured down as the crowd scattered for cover.",
    "The algorithm processed millions of records overnight.",
    "Children laughed and played in the park all afternoon.",
    "The old library held thousands of forgotten books.",
    "Lightning struck the tower during the fierce storm.",
    "The chef prepared an exquisite five-course dinner.",
    "Waves crashed against the rocky shoreline at dawn.",
    "The professor explained quantum mechanics to the class.",
    "A gentle breeze carried the scent of fresh flowers.",
    "The detective examined the evidence under dim light.",
]
raw_acts = sae_store.extract(sae_texts, tokenizer, layers=["h.8"], pooling="none")
act_array = raw_acts["h.8"]  # (n_samples, seq_len, 768)
act_flat = torch.tensor(act_array.reshape(-1, 768), dtype=torch.float32)
print(f"Activations: {act_array.shape} -> flattened to {act_flat.shape}")

# 2. Train SAE
sae = SparseAutoencoder(input_dim=768, feature_dim=2048, k=32)
dataloader = DataLoader(TensorDataset(act_flat), batch_size=128, shuffle=True)
trainer = SAETrainer(sae, dataloader, log_every=50)
stats = trainer.train(n_epochs=3)
print(f"Final MSE: {stats['final_mse']:.4f} | Explained var: {stats['final_explained_variance']:.3f} | Mean L0: {stats['mean_l0']:.1f}")

In [ ]:
from glassboxllms.visualization.plots import plot_sae_training_curves, plot_sae_sparsity

# Training curves
fig1 = plot_sae_training_curves(stats["training_history"], metrics=["loss", "mse_loss"])
plt.show()

# Per-feature sparsity distribution
feature_sparsities = np.array([f["sparsity"] for f in stats["per_feature_stats"]])
fig2 = plot_sae_sparsity(feature_sparsities)
plt.show()

print(f"Dead features (never fired): {(feature_sparsities == 0).sum()} / {len(feature_sparsities)}")

The SAE learns to reconstruct activations using only ~32 of 2048 features per token (TopK sparsity). Most features are rarely active — a sign that each captures a specific, narrow pattern rather than a diffuse blend.

---
## 4. Circuit Discovery — Mapping GPT-2's IOI Circuit

Mechanistic interpretability identifies **circuits** — small subnetworks responsible for specific behaviors. Here we manually construct GPT-2's Indirect Object Identification (IOI) circuit, which handles sentences like *"When Mary and John went to the store, John gave a drink to ___"* and predicts "Mary". We visualize the key attention heads and MLP layers involved.

In [ ]:
from glassboxllms.analysis.circuits import CircuitGraph, NodeType, EdgeType
from glassboxllms.visualization.plots import plot_circuit_graph

# Build GPT-2 IOI circuit: key components from Wang et al. (2022)
ioi = CircuitGraph(model="gpt2", name="IOI Circuit (GPT-2)")

# Embedding input
ioi.add_node("embed", node_type="embedding", layer=0, label="Token Embed")

# Duplicate token heads — detect repeated subject ("John … John")
ioi.add_node("attn.0.h1", node_type="attention_head", layer=0, index=1, label="Dup Token 0.1")
ioi.add_node("attn.3.h0", node_type="attention_head", layer=3, index=0, label="Dup Token 3.0")

# S-Inhibition heads — suppress the repeated name so it isn't predicted
ioi.add_node("attn.7.h3", node_type="attention_head", layer=7, index=3, label="S-Inhib 7.3")
ioi.add_node("attn.8.h6", node_type="attention_head", layer=8, index=6, label="S-Inhib 8.6")

# Name Mover heads — copy the correct (indirect) name to the output
ioi.add_node("attn.9.h9", node_type="attention_head", layer=9, index=9, label="Name Mover 9.9")
ioi.add_node("attn.10.h0", node_type="attention_head", layer=10, index=0, label="Name Mover 10.0")

# MLP layers that refine the signal
ioi.add_node("mlp.8", node_type="mlp_layer", layer=8, label="MLP 8")
ioi.add_node("mlp.9", node_type="mlp_layer", layer=9, label="MLP 9")

# Edges with synthetic but realistic attribution weights
ioi.add_edge("embed", "attn.0.h1", weight=0.45, edge_type="residual")
ioi.add_edge("embed", "attn.3.h0", weight=0.40, edge_type="residual")
ioi.add_edge("attn.0.h1", "attn.7.h3", weight=0.72, edge_type="attention")
ioi.add_edge("attn.3.h0", "attn.7.h3", weight=0.65, edge_type="attention")
ioi.add_edge("attn.3.h0", "attn.8.h6", weight=0.58, edge_type="attention")
ioi.add_edge("attn.7.h3", "mlp.8", weight=0.50, edge_type="direct")
ioi.add_edge("attn.8.h6", "mlp.8", weight=0.47, edge_type="direct")
ioi.add_edge("mlp.8", "attn.9.h9", weight=0.82, edge_type="direct")
ioi.add_edge("mlp.8", "attn.10.h0", weight=0.75, edge_type="direct")
ioi.add_edge("attn.7.h3", "mlp.9", weight=0.38, edge_type="direct")
ioi.add_edge("mlp.9", "attn.9.h9", weight=0.55, edge_type="direct")
ioi.add_edge("mlp.9", "attn.10.h0", weight=0.48, edge_type="direct")

# Visualize
fig = plot_circuit_graph(ioi, layout="layer", figsize=(14, 8), node_size=1000, font_size=8)
fig.suptitle("GPT-2 IOI Circuit — Indirect Object Identification", fontsize=14, y=1.02)
plt.show()

print(ioi.summary())

The IOI circuit reveals a clean **three-stage pipeline**: duplicate-token heads detect the repeated name, S-inhibition heads suppress it, and name-mover heads copy the *other* name to the output. Just 9 nodes and 12 edges out of GPT-2's full 144-head network are enough to explain the behavior.

---
## 5. Logit Lens — Watching GPT-2 Converge on a Prediction

The logit lens projects each layer's hidden state through the final unembedding matrix to see what the model would predict *if decoding stopped at that layer*. Early layers produce near-random guesses; later layers sharpen toward the final answer.

In [ ]:
from glassboxllms.visualization.plots import plot_logit_lens

text = "The capital of France is"
inputs = tokenizer(text, return_tensors="pt")
tokens = [tokenizer.decode(t) for t in inputs["input_ids"][0]]

with torch.no_grad():
    outputs = lm_model(**inputs, output_hidden_states=True)

# hidden_states: tuple of (n_layers+1) tensors, each (1, seq_len, 768)
hidden_states = outputs.hidden_states[1:]  # skip embedding, keep 12 layers
n_layers = len(hidden_states)
seq_len = len(tokens)

# For each layer, project through lm_head -> softmax -> track correct-next-token prob
correct_token_ids = inputs["input_ids"][0][1:]
logit_lens_data = np.zeros((n_layers, seq_len))
top_k_tokens = []

for layer_idx, hs in enumerate(hidden_states):
    logits = lm_model.lm_head(hs[0])
    probs = torch.softmax(logits, dim=-1)

    layer_top_k = []
    for pos in range(seq_len):
        top_ids = probs[pos].topk(3).indices.tolist()
        layer_top_k.append([tokenizer.decode(t) for t in top_ids])

        if pos < seq_len - 1:
            logit_lens_data[layer_idx, pos] = probs[pos, correct_token_ids[pos]].item()
        else:
            logit_lens_data[layer_idx, pos] = probs[pos].max().item()

    top_k_tokens.append(layer_top_k)

fig = plot_logit_lens(logit_lens_data, tokens, top_k_tokens=top_k_tokens)
fig.suptitle('Logit Lens: "The capital of France is" (GPT-2)', fontsize=14, y=1.02)
plt.show()
print(f"Final layer top prediction for last token: {top_k_tokens[-1][-1][0]!r}")

The heatmap shows correct-next-token probability climbing across layers — early layers are essentially guessing, while the final layers concentrate probability mass on the right answer (e.g., "Paris"). The cell annotations reveal *when* the model commits to its prediction.

---
## 6. Steering Vectors — Pushing GPT-2's Sentiment

A steering vector is the difference in mean activations between positive and negative examples at a chosen layer. By adding scaled copies of this vector during inference, we can shift the model's internal representation toward positive or negative sentiment without retraining.

In [ ]:
from glassboxllms.primitives.probes.activation_store import ActivationStore
from glassboxllms.visualization.plots import plot_steering_effects

# --- 1. Compute steering vector at the best probing layer ---
steer_store = ActivationStore(model)
target_layer = "h.10"  # late layer where sentiment is linearly encoded

pos_texts = [
    "I absolutely love this movie, it was fantastic!",
    "This is the best day of my life, I'm so happy!",
    "What a wonderful experience, highly recommended!",
    "The food was delicious and the service was excellent.",
    "Amazing performance, the actors were brilliant!",
]
neg_texts = [
    "I hate this movie, it was terrible and boring.",
    "This is the worst day ever, everything went wrong.",
    "What a horrible experience, never going back!",
    "The food was disgusting and the service was awful.",
    "Terrible performance, the actors were embarrassing.",
]

pos_acts = steer_store.extract(pos_texts, tokenizer, layers=[target_layer], pooling="mean")[target_layer]
neg_acts = steer_store.extract(neg_texts, tokenizer, layers=[target_layer], pooling="mean")[target_layer]

steering_vector = pos_acts.mean(axis=0) - neg_acts.mean(axis=0)
print(f"Steering vector at {target_layer}: norm={np.linalg.norm(steering_vector):.2f}, dims={steering_vector.shape[0]}")
print(f"Top-5 dimensions: {np.argsort(np.abs(steering_vector))[-5:][::-1]}")

# --- 2. Measure effect: project a neutral sentence onto the vector at each scale ---
neutral = "The movie was okay and had some interesting parts."
neutral_act = steer_store.extract([neutral], tokenizer, layers=[target_layer], pooling="mean")[target_layer][0]

sv_unit = steering_vector / np.linalg.norm(steering_vector)
base_proj = float(np.dot(neutral_act, sv_unit))

effects = {}
for alpha in [-2.0, -1.0, 0.0, 1.0, 2.0]:
    steered = neutral_act + alpha * steering_vector
    proj = float(np.dot(steered, sv_unit))
    effects[f"{alpha:+.0f}x"] = {"sentiment_projection": proj, "shift": proj - base_proj}

print("\nSteering effects (sentiment projection):")
for k, v in effects.items():
    print(f"  {k}: projection={v['sentiment_projection']:.2f}  shift={v['shift']:+.2f}")

# --- 3. Visualize ---
fig = plot_steering_effects(effects, metric="sentiment_projection")
fig.suptitle(f"Steering Vector Effect on Sentiment ({target_layer})", fontsize=14, y=1.02)
plt.show()

The sentiment projection scales linearly with steering strength, confirming the vector captures a clean linear direction. This is the same direction the probe learned to read in Section 1 — steering writes to it, probing reads from it.

---
## 7. Causal Patching — Which Layers Drive Name Prediction?

Causal patching swaps a layer's activation from a clean run into a corrupted run. If the answer shifts back toward the clean answer, that layer causally carries the signal. We test GPT-2's Indirect Object Identification: *"When Mary and John went to the store, John gave a drink to ___"* (answer: Mary).

In [ ]:
import torch.nn.functional as F
from operator import attrgetter

source_prompt = "When Mary and John went to the store, John gave a drink to"
corrupted_prompt = "When Mary and John went to the store, Mary gave a drink to"
target_token = " Mary"
target_id = tokenizer.encode(target_token)[-1]

corr_ids = tokenizer(corrupted_prompt, return_tensors="pt").input_ids
clean_ids = tokenizer(source_prompt, return_tensors="pt").input_ids

with torch.no_grad():
    baseline_p = F.softmax(lm_model(corr_ids).logits[0, -1], dim=-1)[target_id].item()
    clean_p = F.softmax(lm_model(clean_ids).logits[0, -1], dim=-1)[target_id].item()
print(f'P("{target_token.strip()}") clean={clean_p:.3f}, corrupted={baseline_p:.3f}\n')

# Patch each MLP and measure how much P(Mary) is restored
effects = []
for i in range(model.config.n_layer):
    mod = attrgetter(f"transformer.h.{i}.mlp")(lm_model)
    cache = {}
    def cache_hook(module, input, output, c=cache):
        c['act'] = output.detach().clone()
        return output
    h = mod.register_forward_hook(cache_hook)
    with torch.no_grad(): lm_model(clean_ids)
    h.remove()

    def patch_hook(module, input, output, c=cache):
        return c['act']
    h = mod.register_forward_hook(patch_hook)
    with torch.no_grad(): patched_logits = lm_model(corr_ids).logits
    h.remove()

    patched_p = F.softmax(patched_logits[0, -1], dim=-1)[target_id].item()
    ie = patched_p - baseline_p
    effects.append(ie)
    print(f"  mlp.{i}: P(Mary)={patched_p:.3f}, IE={ie:+.4f}")

print(f"\nStrongest MLP: layer {np.argmax(np.abs(effects))} (IE = {effects[np.argmax(np.abs(effects))]:+.4f})")

In [ ]:
# Bar chart of MLP indirect effects across layers
fig, ax = plt.subplots(figsize=(12, 4))
colors = ["#c44e52" if e > 0 else "#4c72b0" for e in effects]
ax.bar(range(len(effects)), effects, color=colors, edgecolor="black", linewidth=0.5)
ax.set_xlabel("Layer")
ax.set_ylabel("Indirect Effect (Δ P(Mary))")
ax.set_title('Causal Patching: Which MLP layers recover P(" Mary") in the IOI task?')
ax.set_xticks(range(len(effects)))
ax.set_xticklabels([str(i) for i in range(len(effects))])
ax.axhline(0, color="black", linewidth=0.8)
fig.tight_layout()
plt.show()

MLP 0 shows the strongest causal effect — it writes the name information into the residual stream early. Later MLPs have small positive or negative contributions, showing the distributed nature of the computation.

---
## 8. Attention Patterns — What Does GPT-2 Focus On?

Attention heatmaps reveal which tokens each head attends to when processing a sentence. By comparing an early-layer head (layer 0) with a late-layer head (layer 11), we can see how attention shifts from local, positional patterns to long-range, semantic dependencies.

In [ ]:
from glassboxllms.visualization.plots import plot_attention_heatmap

text = "The cat sat on the mat because it was tired"
inputs = tokenizer(text, return_tensors="pt")
tokens = [tokenizer.decode(t) for t in inputs["input_ids"][0]]

with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)

# outputs.attentions is a tuple of (batch, heads, seq, seq) per layer
# Extract head 0 from an early layer and a late layer
attn_early = outputs.attentions[0][0, 0].numpy()   # layer 0, head 0
attn_late = outputs.attentions[11][0, 0].numpy()    # layer 11, head 0

# Side-by-side heatmaps
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, attn, title in [
    (axes[0], attn_early, "Layer 0, Head 0 (early)"),
    (axes[1], attn_late, "Layer 11, Head 0 (late)"),
]:
    sns.heatmap(attn, ax=ax, cmap="viridis", xticklabels=tokens,
                yticklabels=tokens, square=True, vmin=0, vmax=attn.max())
    ax.set_title(title)
    ax.set_xlabel("Key position")
    ax.set_ylabel("Query position")

fig.suptitle("Attention Patterns: Early vs Late Layer (GPT-2)", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

# Individual detailed view using the library helper
fig_detail = plot_attention_heatmap(attn_late, tokens, layer=11, head=0)
plt.show()

The early-layer head shows a strong diagonal pattern (each token attending to itself or its immediate neighbor), while the late-layer head spreads attention across semantically relevant tokens — for example, "it" attending back to "cat". This confirms that GPT-2 builds local context first and resolves long-range dependencies in deeper layers.

---
## 9. Animated Visualizations (Manim)

The glassbox-llms visualization module includes cinematic Manim animations that bring interpretability concepts to life. Below are pre-rendered animations using real data from the experiments above.

In [ ]:
from IPython.display import Video, display

# Probing: scatter with decision boundary
print("Probing Hyperplane — Sentiment classification in activation space")
try:
    display(Video("../media/demo/ProbingHyperplaneScene.mp4", embed=True, width=800))
except FileNotFoundError:
    print("  (Pre-render with: python scripts/render_demo_animations.py)")

In [ ]:
# Circuit Discovery: Animated IOI circuit graph
print("Circuit Discovery — GPT-2 IOI Circuit")
try:
    display(Video("../media/demo/CircuitDiscoveryScene.mp4", embed=True, width=800))
except FileNotFoundError:
    print("  (Pre-render with: python scripts/render_demo_animations.py)")

In [ ]:
# Steering: Before/after activation shift
print("Steering Vector — Sentiment direction in activation space")
try:
    display(Video("../media/demo/SteeringVectorScene.mp4", embed=True, width=800))
except FileNotFoundError:
    print("  (Pre-render with: python scripts/render_demo_animations.py)")

These animations are generated from the same real GPT-2 experiment data shown in the static plots above, converted through the `glassboxllms.visualization.adapters` layer into Manim scene data.